In [1]:
import contextlib
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
import wandb
from tqdm.autonotebook import tqdm

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.experiments.multimodal_gaussian.canonical import (
    get_canonical_chart_transport_configs,
    get_canonical_chart_transport_monitor_configs,
)
from src.experiments.multimodal_gaussian.config import MultimodalGaussianTrainingConfig
from src.experiments.multimodal_gaussian.state import MultimodalTrainingRuntime

/tmp/ipykernel_3415257/587479404.py:11: TqdmExperimentalWarning:

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)



In [2]:
num_modes = 8
latent_dimension = 128
ambient_dimension = 128
experiment_name = f"{datetime.now():%m%d%H%M%S}-{uuid4().hex[:6]}"
wandb_project = "diffusive-latent-generation"

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=0.1,
    offset=0.0,
    ambient_dimension=ambient_dimension,
)

training_config = MultimodalGaussianTrainingConfig.initialize(
    seed=0,
    train_batch_size=4096,
    eval_batch_size=4096,
    integrated_n_steps=1_000_000,
    chart_transport_config=get_canonical_chart_transport_configs(
        data_config=data_config,
        latent_dimension=latent_dimension,
    ),
    monitor_config=get_canonical_chart_transport_monitor_configs(),
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian")
        / experiment_name
    ),
)

training_config.visualize()

In [3]:
cuda_device = 1
runtime = MultimodalTrainingRuntime.initialize(
    tc=training_config,
    cuda_device=cuda_device,
    wandb_project=wandb_project,
    run_name=experiment_name,
)

Using bfloat16 Automatic Mixed Precision (AMP)
Seed set to 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nlyu/.netrc.
wandb: Currently logged in as: lyuxingjian (lyuxingjian-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Pretrain

Chart pretrain is now delegated to the runtime. It logs constraint and conditioning monitors on the configured schedule.


In [4]:
latest_pretrain_metrics = runtime.chart_pretrain()
latest_pretrain_metrics

pretrain:   0%|          | 0/2000 [00:00<?, ?it/s]

[pretrain] step 1/2000: train: chart_loss=0.6208, data_cycle_loss=0.0545, prior_cycle_loss=0.5657, zero_mean_loss=0.0000, latent_norm_loss=5.6209; monitor: constraint_reconstruction_mean=5.8886, constraint_reconstruction_max=6.3288, constraint_latent_norm_mean=9.3096, encoder_conditioning_mean=9.9209, encoder_conditioning_max=15.2271, decoder_conditioning_mean=1.1243, decoder_conditioning_max=1.2225
[pretrain] step 2000/2000: train: chart_loss=0.0010, data_cycle_loss=0.0003, prior_cycle_loss=0.0006, zero_mean_loss=0.0000, latent_norm_loss=0.6760; monitor: constraint_reconstruction_mean=0.2489, constraint_reconstruction_max=0.2821, constraint_latent_norm_mean=1.1578, encoder_conditioning_mean=6.5668, encoder_conditioning_max=14.1973, decoder_conditioning_mean=1.4964, decoder_conditioning_max=1.7569


{'train_chart_loss': 0.0009635469759814441,
 'train_data_cycle_loss': 0.00029723800253123045,
 'train_prior_cycle_loss': 0.0005987091572023928,
 'train_zero_mean_loss': 1.6153353499248624e-07,
 'train_latent_norm_loss': 0.6759979724884033,
 'monitor_constraint_reconstruction_mean': 0.24886350333690643,
 'monitor_constraint_reconstruction_max': 0.28206124901771545,
 'monitor_constraint_latent_norm_mean': 1.1577767133712769,
 'monitor_encoder_conditioning_mean': 6.566794395446777,
 'monitor_encoder_conditioning_max': 14.197318077087402,
 'monitor_decoder_conditioning_mean': 1.4964169263839722,
 'monitor_decoder_conditioning_max': 1.7569286823272705}

## Train noise critic

In [ ]:
latest_critic_pretrain_metrics = runtime.critic_pretrain()
latest_critic_pretrain_metrics

critic_pretrain:   0%|          | 0/2000 [00:00<?, ?it/s]

[critic_pretrain] step 1/2000: train: critic_loss=1.1200; monitor: critic_monitor_snapshot_score_norm_mean=104.0885, critic_monitor_transport_norm_mean=24.9953, critic_monitor_transport_norm_max=26.0053, critic_monitor_transport_t_min=0.0300, critic_monitor_transport_t_max=0.9500
[critic_pretrain] step 2000/2000: train: critic_loss=0.0063; monitor: critic_monitor_snapshot_score_norm_mean=206.1602, critic_monitor_transport_norm_mean=4.3433, critic_monitor_transport_norm_max=6.9120, critic_monitor_transport_t_min=0.0300, critic_monitor_transport_t_max=0.9500


{'train_critic_loss': 0.0063293976709246635,
 'monitor_critic_monitor_snapshot_score_norm_mean': 206.16021728515625,
 'monitor_critic_monitor_transport_norm_mean': 4.343270301818848,
 'monitor_critic_monitor_transport_norm_max': 6.911968231201172,
 'monitor_critic_monitor_transport_t_min': 0.029999999329447746,
 'monitor_critic_monitor_transport_t_max': 0.949999988079071}

## Integrated training

In [ ]:
latest_integrated_metrics = None
critic_steps_per_chart_step = max(
    1,
    chart_transport_config.scheduling_config.update_chart_every_n_critic_steps,
)
latest_chart_metrics = {
    "chart_loss": float("nan"),
    "encoder_transport_loss": float("nan"),
    "decoder_transport_loss": float("nan"),
    "transport_field_norm": float("nan"),
    "avg_generated_log_likelihood": float("nan"),
    "data_cycle_loss": float("nan"),
    "prior_cycle_loss": float("nan"),
}

integrated_progress = tqdm(
    range(1, training_config.integrated_n_steps + 1),
    desc="integrated",
)
for step in integrated_progress:
    encoder.eval()
    decoder.eval()
    critic.train()
    optimizer.zero_grad(set_to_none=True)

    with runtime_precision_context(rt=runtime):
        critic_data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            critic_data_latents = encoder(critic_data_batch)

        t_min, t_max = transport_config.t_range
        critic_t = t_min + (t_max - t_min) * torch.rand(
            critic_data_latents.shape[0],
            device=device,
            dtype=torch.float32,
        )
        critic_eps = torch.randn_like(critic_data_latents)
        critic_noised_latents = (
            (1.0 - critic_t).unsqueeze(-1) * critic_data_latents
            + critic_t.unsqueeze(-1) * critic_eps
        )
        critic_predicted_noise = critic(critic_noised_latents, critic_t)
        critic_loss = F.mse_loss(critic_predicted_noise, critic_eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "integrated",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
        **latest_chart_metrics,
        "data_dual": runtime.data_dual.detach().item(),
        "prior_dual": runtime.prior_dual.detach().item(),
    }
    wandb_metrics = {
        "critic_loss": critic_loss.detach().item(),
        "data_dual": runtime.data_dual.detach().item(),
        "prior_dual": runtime.prior_dual.detach().item(),
    }

    if step == 1 or step % critic_steps_per_chart_step == 0:
        encoder.train()
        decoder.train()
        optimizer.zero_grad(set_to_none=True)

        with runtime_precision_context(rt=runtime):
            chart_data_batch = runtime_data_config.sample_unconditional(
                batch_size=training_config.train_batch_size,
            )
            chart_prior_batch = prior_config.sample(
                batch_size=training_config.train_batch_size,
            ).to(device=device, dtype=torch.float32)

            chart_data_latents = encoder(chart_data_batch)
            chart_data_reconstruction = decoder(chart_data_latents)
            chart_prior_reconstruction = decoder(chart_prior_batch)
            chart_prior_latents = encoder(chart_prior_reconstruction)

            data_cycle_loss = F.huber_loss(
                chart_data_reconstruction,
                chart_data_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )
            prior_cycle_loss = F.huber_loss(
                chart_prior_latents,
                chart_prior_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )

            if isinstance(constraint_method, LossConstraintConfig):
                constraint_loss = (
                    constraint_method.data_loss_weight * data_cycle_loss
                    + constraint_method.prior_loss_weight * prior_cycle_loss
                )
            else:
                constraint_loss = data_cycle_loss + prior_cycle_loss
                constraint_loss = constraint_loss + runtime.data_dual * (
                    data_cycle_loss - constraint_method.data_constraint_budget
                )
                constraint_loss = constraint_loss + runtime.prior_dual * (
                    prior_cycle_loss - constraint_method.prior_constraint_budget
                )

            with torch.no_grad():
                transport_source_latents = chart_data_latents.detach()
                t_min, t_max = transport_config.t_range
                stratified_offsets = torch.rand(
                    transport_source_latents.shape[0],
                    transport_config.num_time_samples,
                    device=device,
                    dtype=transport_source_latents.dtype,
                )
                bin_left = torch.arange(
                    transport_config.num_time_samples,
                    device=device,
                    dtype=transport_source_latents.dtype,
                ) / transport_config.num_time_samples
                transport_t = t_min + (t_max - t_min) * (
                    bin_left.unsqueeze(0) + stratified_offsets
                ) / transport_config.num_time_samples

                transport_source_latents = transport_source_latents.unsqueeze(1).expand(
                    -1,
                    transport_config.num_time_samples,
                    -1,
                )
                transport_eps = torch.randn(
                    transport_source_latents.shape[0],
                    transport_config.num_time_samples,
                    transport_source_latents.shape[-1],
                    device=device,
                    dtype=transport_source_latents.dtype,
                )

                if transport_config.antipodal_estimate:
                    transport_t = torch.cat([transport_t, transport_t], dim=1)
                    transport_source_latents = transport_source_latents.repeat(1, 2, 1)
                    transport_eps = torch.cat([transport_eps, -transport_eps], dim=1)

                transport_noised_latents = (
                    (1.0 - transport_t).unsqueeze(-1) * transport_source_latents
                    + transport_t.unsqueeze(-1) * transport_eps
                )
                flat_transport_noised_latents = transport_noised_latents.reshape(
                    -1,
                    transport_noised_latents.shape[-1],
                )
                flat_transport_t = transport_t.reshape(-1)

                transport_predicted_noise = critic(
                    flat_transport_noised_latents,
                    flat_transport_t,
                ).reshape_as(transport_noised_latents)
                transport_prior_score = prior_config.analytic_score(
                    flat_transport_noised_latents.float(),
                    flat_transport_t.float(),
                ).to(dtype=flat_transport_noised_latents.dtype).reshape_as(
                    transport_noised_latents
                )
                transport_pullback_weight = (
                    transport_config.kl_weight_schedule.pullback_weight(
                        flat_transport_t.float(),
                    )
                    .to(dtype=flat_transport_noised_latents.dtype)
                    .reshape_as(transport_t)
                )
                transport_field_terms = transport_pullback_weight.unsqueeze(-1) * (
                    transport_prior_score
                    + transport_predicted_noise / transport_t.unsqueeze(-1)
                )

                if transport_config.antipodal_estimate:
                    midpoint = transport_config.num_time_samples
                    transport_field_terms = 0.5 * (
                        transport_field_terms[:, :midpoint]
                        + transport_field_terms[:, midpoint:]
                    )

                transport_field = transport_field_terms.mean(dim=1)
                transport_field_norm = transport_field.norm(dim=-1, keepdim=True)
                transport_step_size = torch.minimum(
                    torch.full_like(
                        transport_field_norm,
                        transport_config.transport_step_size,
                    ),
                    torch.full_like(
                        transport_field_norm,
                        transport_config.transport_step_cap,
                    )
                    / transport_field_norm.clamp_min(1e-6),
                )
                transport_step = transport_step_size * transport_field
                transported_latents = chart_data_latents.detach() + transport_step

            encoder_transport_loss = F.mse_loss(
                chart_data_latents,
                transported_latents.detach(),
            )
            decoder_transport_loss = F.mse_loss(
                decoder(transported_latents),
                chart_data_batch.detach(),
            )
            generated_log_likelihood = runtime_data_config.log_likelihood(
                chart_prior_reconstruction.float(),
            ).mean()
            chart_loss = constraint_loss
            chart_loss = chart_loss + (
                transport_config.encoder_transport_weight * encoder_transport_loss
            )
            chart_loss = chart_loss + (
                transport_config.decoder_transport_weight * decoder_transport_loss
            )

        fabric.backward(chart_loss)
        fabric.clip_gradients(
            chart_transport_model,
            optimizer,
            max_norm=architecture_config.grad_clip_norm,
        )
        optimizer.step()

        if isinstance(constraint_method, LagrangianConstraintConfig):
            runtime.data_dual = (
                runtime.data_dual
                + constraint_method.dual_variable_lr
                * (data_cycle_loss.detach() - constraint_method.data_constraint_budget)
            ).clamp_min(0.0)
            runtime.prior_dual = (
                runtime.prior_dual
                + constraint_method.dual_variable_lr
                * (prior_cycle_loss.detach() - constraint_method.prior_constraint_budget)
            ).clamp_min(0.0)

        latest_chart_metrics = {
            "chart_loss": chart_loss.detach().item(),
            "encoder_transport_loss": encoder_transport_loss.detach().item(),
            "decoder_transport_loss": decoder_transport_loss.detach().item(),
            "transport_field_norm": transport_field_norm.mean().detach().item(),
            "avg_generated_log_likelihood": generated_log_likelihood.detach().item(),
            "data_cycle_loss": data_cycle_loss.detach().item(),
            "prior_cycle_loss": prior_cycle_loss.detach().item(),
        }
        metrics.update(latest_chart_metrics)
        metrics["data_dual"] = runtime.data_dual.detach().item()
        metrics["prior_dual"] = runtime.prior_dual.detach().item()
        wandb_metrics.update(latest_chart_metrics)
        wandb_metrics["data_dual"] = runtime.data_dual.detach().item()
        wandb_metrics["prior_dual"] = runtime.prior_dual.detach().item()

    latest_integrated_metrics = metrics
    log_wandb_scalars(rt=runtime, stage="integrated", step=step, metrics=wandb_metrics)
    integrated_progress.set_postfix(
        critic_loss=f"{metrics['critic_loss']:.4f}",
        chart_loss=f"{metrics['chart_loss']:.4f}",
    )

    run_constraint_monitor = monitor_config.constraint_monitor_config.should_activate(
        step=step,
        total_steps=training_config.integrated_n_steps,
        every_n_steps=monitor_config.schedule_config.log_every_n_steps_integrated,
    )
    run_conditioning_monitor = monitor_config.conditioning_monitor_config.should_activate(
        step=step,
        total_steps=training_config.integrated_n_steps,
        every_n_steps=monitor_config.schedule_config.log_every_n_steps_integrated,
    )
    run_critic_monitor = monitor_config.critic_monitor_config.should_activate(
        step=step,
        total_steps=training_config.integrated_n_steps,
        every_n_steps=monitor_config.schedule_config.log_every_n_steps_integrated,
    )
    run_sampling_monitor = monitor_config.sampling_monitor_config.should_activate(
        step=step,
        total_steps=training_config.integrated_n_steps,
        every_n_steps=monitor_config.schedule_config.log_every_n_steps_integrated,
    )

    if (
        run_constraint_monitor
        or run_conditioning_monitor
        or run_critic_monitor
        or run_sampling_monitor
    ):
        monitor_metrics: dict[str, float] = {}
        with torch.no_grad():
            with runtime_precision_context(rt=runtime):
                if run_constraint_monitor:
                    monitor_metrics.update(
                        monitor_config.constraint_monitor_config.apply_to(
                            rt=runtime,
                            step=step,
                        )
                    )
                if run_conditioning_monitor:
                    monitor_metrics.update(
                        monitor_config.conditioning_monitor_config.apply_to(
                            rt=runtime,
                            step=step,
                        )
                    )
                if run_critic_monitor:
                    monitor_metrics.update(
                        monitor_config.critic_monitor_config.apply_to(
                            rt=runtime,
                            step=step,
                            stage="integrated",
                        )
                    )
                if run_sampling_monitor:
                    monitor_metrics.update(
                        apply_sampling_monitor(
                            rt=runtime,
                            step=step,
                        )
                    )

        log_wandb_scalars(
            rt=runtime,
            stage="integrated",
            step=step,
            metrics={f"monitor_{key}": value for key, value in monitor_metrics.items()},
        )
        monitor_summary_metrics = {
            key: value for key, value in monitor_metrics.items() if "_mode_" not in key
        }
        fabric.print(
            f"[integrated] step {step}/{training_config.integrated_n_steps}: "
            f"train: {format_metrics_summary(metrics=wandb_metrics)}; "
            f"monitor: {format_metrics_summary(metrics=monitor_summary_metrics)}"
        )
        latest_integrated_metrics = {
            **metrics,
            **{f"monitor_{key}": value for key, value in monitor_metrics.items()},
        }

wandb.finish()
latest_integrated_metrics
